# Notebook 07 FMM-inspired energy encoder

Reusable definitions for a conditional quadratic energy encoder. The context branch uses only visible raw patches; the EMA teacher uses the complete image; the existing JEPA predictor and SmoothL1 loss remain unchanged. The model combines an anchored patch state, full-width local diagonal-affine factors, and quadtree cell-mean factors of width 32. Coefficients are nonlinear functions of the initial observed embeddings and remain fixed throughout four refinement steps.

With nonnegative strengths, degree-normalized weights, diagonal contractions and an exactly spectrally normalized summary projection, the energy Hessian lies between I and (1+2 lambda_near+2 lambda_far) I. At both strengths 0.25 and step 0.5, four steps contract distance to the unique optimum by at least 16x in exact arithmetic. This is an inference guarantee, not a guarantee of useful learned representations.

Inputs: images, original patch indices and validity masks. Outputs: ordered per-patch features and optional energy diagnostics. Expected definition-only runtime under 10 seconds. No dataset access, training, checkpoints or background work occurs here. Restart the `ahr-go` kernel and Run All to reload definitions. The Notebook 07 experiment owns checkpoints and exact training resume instructions. Import only `ahr-export` cells.

The `anchor` control uses the same patch stem and normalization with no refinement. It keeps otherwise unused energy parameters for state alignment, so parameter reports must distinguish total and active parameters. FP32 energy inference is deliberate; the predictor may use the surrounding mixed-precision policy. Geometry is cached by grid and device; padded and empty cells are excluded from all factors.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_namespace = {}
for _cell in json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _namespace)
_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""A conditional convex energy encoder with local and hierarchical cell factors."""
import copy
import math
from functools import lru_cache
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from fmm_jepa.models import JEPA, Predictor, position_features, validate_positions

BACKEND = "fmm-energy-v1"
_DEVICE_GEOMETRY = {}


@lru_cache(maxsize=32)
def cpu_geometry(grid, leaf_size=2):
    """Canonical unordered edges; nonadjacent cells whose parents are adjacent."""
    rows, cols = map(int, grid)
    side = 1 << (max(math.ceil(rows / leaf_size), math.ceil(cols / leaf_size)) - 1).bit_length()
    near_i, near_j, near_xy = [], [], []
    for i in range(rows * cols):
        row, col = divmod(i, cols)
        lr, lc = row // leaf_size, col // leaf_size
        for rr in range(max(0, (lr-1)*leaf_size), min(rows, (lr+2)*leaf_size)):
            for cc in range(max(0, (lc-1)*leaf_size), min(cols, (lc+2)*leaf_size)):
                j = rr * cols + cc
                if i < j:
                    near_i.append(i); near_j.append(j)
                    near_xy.append(((cc-col)/leaf_size, (rr-row)/leaf_size))
    levels = []
    n = 1
    while n <= side:
        ei, ej, rel = [], [], []
        for row in range(n):
            for col in range(n):
                i = row*n+col
                for pr in range(max(0,row//2-1), min(n//2,row//2+2)):
                    for pc in range(max(0,col//2-1), min(n//2,col//2+2)):
                        for dr in (0,1):
                            for dc in (0,1):
                                rr, cc = 2*pr+dr, 2*pc+dc
                                j = rr*n+cc
                                if i < j and max(abs(rr-row),abs(cc-col)) > 1:
                                    ei.append(i); ej.append(j); rel.append((cc-col,rr-row))
        parent = torch.tensor([(r//2)*(n//2)+c//2 for r in range(n) for c in range(n)], dtype=torch.long) if n>1 else None
        levels.append({"side":n,"i":torch.tensor(ei,dtype=torch.long),"j":torch.tensor(ej,dtype=torch.long),
                       "xy":torch.tensor(rel,dtype=torch.float32).reshape(-1,2),"parent":parent})
        n *= 2
    cells = torch.tensor([(i//cols//leaf_size)*side + (i%cols//leaf_size) for i in range(rows*cols)],dtype=torch.long)
    return {"grid":tuple(grid),"side":side,"cells":cells,"levels":levels,
            "near_i":torch.tensor(near_i,dtype=torch.long),"near_j":torch.tensor(near_j,dtype=torch.long),
            "near_xy":torch.tensor(near_xy,dtype=torch.float32).reshape(-1,2)}


def device_geometry(grid, leaf_size, device):
    key = (tuple(grid), int(leaf_size), str(device))
    if key not in _DEVICE_GEOMETRY:
        raw = cpu_geometry(tuple(grid), int(leaf_size))
        result = {k:(v.to(device) if torch.is_tensor(v) else v) for k,v in raw.items() if k != "levels"}
        result["levels"] = [{k:(v.to(device) if torch.is_tensor(v) else v) for k,v in level.items()} for level in raw["levels"]]
        _DEVICE_GEOMETRY[key] = result
    return _DEVICE_GEOMETRY[key]


def _gather(values, indices):
    if indices.ndim == 1:
        return values.index_select(1, indices)
    return values.gather(1, indices[...,None].expand(-1,-1,values.shape[-1]))


def _scatter(values, indices, count):
    if indices.ndim == 1:
        indices = indices[None,:].expand(values.shape[0],-1)
    return values.new_zeros(values.shape[0],count,values.shape[-1]).scatter_add(1,indices[...,None].expand_as(values),values)


def _degree_weights(i, j, active, count, dtype=torch.float32):
    # Degrees include both endpoints, so every weighted incident sum is <=1.
    degrees = _scatter(active[...,None].to(dtype),i,count) + _scatter(active[...,None].to(dtype),j,count)
    denominator = torch.maximum(_gather(degrees,i), _gather(degrees,j)).clamp_min(1)
    return active[...,None].to(dtype) / denominator


class EnergyEncoder(nn.Module):
    """Four analytic descent steps; coefficients depend on h but never on z."""
    def __init__(self, patch_size=8, dim=192, summary_dim=32, leaf_size=2,
                 refinement_steps=4, near_strength=.25, far_strength=.25,
                 step_size=.5, coefficient_dim=32, checkpoint_refinement=True,
                 edge_tile=256, variant="energy"):
        super().__init__()
        if min(patch_size,dim,summary_dim,leaf_size,refinement_steps,coefficient_dim,edge_tile)<1:
            raise ValueError("Positive model dimensions and refinement count required")
        if variant not in ("energy","anchor"):
            raise ValueError("variant must be energy or anchor")
        if min(near_strength,far_strength)<0:
            raise ValueError("Energy strengths must be nonnegative")
        if not 0 < step_size <= 1/(1+2*near_strength+2*far_strength):
            raise ValueError("step_size exceeds the certified monotone contraction bound")
        self.patch_size, self.dim, self.kind = int(patch_size), int(dim), "energy"
        self.summary_dim,self.leaf_size=int(summary_dim),int(leaf_size)
        self.refinement_steps,self.step_size=int(refinement_steps),float(step_size)
        self.near_strength,self.far_strength=float(near_strength),float(far_strength)
        self.checkpoint_refinement,self.edge_tile=bool(checkpoint_refinement),int(edge_tile)
        self.variant=variant
        self.patch_embed=nn.Linear(3*patch_size*patch_size,dim)
        self.norm=nn.LayerNorm(dim)
        self.blocks=nn.ModuleList()  # JEPA's existing optional branch-metric loop.
        self.summary_raw=nn.Parameter(torch.randn(summary_dim,dim)/math.sqrt(dim))
        self.near_coeff=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,coefficient_dim),nn.GELU(),nn.Linear(coefficient_dim,2*dim+1))
        self.far_coeff=nn.Sequential(nn.LayerNorm(summary_dim),nn.Linear(summary_dim,coefficient_dim),nn.GELU(),nn.Linear(coefficient_dim,2*summary_dim+1))
        self.near_position=nn.Linear(2,2*dim+1)
        self.far_position=nn.Linear(2,2*summary_dim+1)
        for layer in (self.near_coeff[-1],self.far_coeff[-1],self.near_position,self.far_position):
            nn.init.normal_(layer.weight,std=.02)
            nn.init.zeros_(layer.bias)

    def summary_projection(self):
        # Exact largest singular value, not an estimated power-iteration norm.
        raw=self.summary_raw
        norm=torch.linalg.matrix_norm(raw,ord=2)
        return raw/norm.clamp_min(1)

    def _tree_sums(self, values, state):
        geometry=state["geometry"]
        sums=[_scatter(values,state["leaf_index"],geometry["side"]**2)]
        for level in reversed(geometry["levels"][1:]):
            sums.append(_scatter(sums[-1],level["parent"],(level["side"]//2)**2))
        return list(reversed(sums))

    def prepare_energy(self, h, indices, valid, grid):
        """Build differentiable fixed conditioning; geometry depends only on masks."""
        b,v,_=h.shape
        geometry=device_geometry(tuple(grid),self.leaf_size,h.device)
        full_count=int(grid[0])*int(grid[1])
        # The sentinel prevents padding from overwriting a valid index 0.
        slots=torch.where(valid,indices,full_count)
        lookup=torch.full((b,full_count+1),v,device=h.device,dtype=torch.long)
        values=torch.arange(v,device=h.device)[None,:].expand(b,-1)
        lookup=lookup.scatter(1,slots,values)
        ni=lookup[:,geometry["near_i"]];nj=lookup[:,geometry["near_j"]]
        near_active=(ni<v)&(nj<v)
        ni=ni.clamp_max(v-1);nj=nj.clamp_max(v-1)
        leaf_index=geometry["cells"][indices.clamp(0,full_count-1)]
        state={"h":h,"valid":valid,"indices":indices,"geometry":geometry,"leaf_index":leaf_index,
               "near_i":ni,"near_j":nj,"near_base":_degree_weights(ni,nj,near_active,v,h.dtype),
               "near_node":self.near_coeff(h),"near_position":self.near_position(geometry["near_xy"].to(h.dtype)),
               "projection":self.summary_projection()}
        state["counts"]=self._tree_sums(valid[...,None].to(h.dtype),state)
        hsums=self._tree_sums(F.linear(h,state["projection"])*valid[...,None],state)
        state["far"]=[]
        num_interacting=sum(int(level["i"].numel()>0) for level in geometry["levels"])
        for level,count,hsum in zip(geometry["levels"],state["counts"],hsums):
            i,j=level["i"],level["j"]
            active=(_gather(count,i)[...,0]>0)&(_gather(count,j)[...,0]>0)
            state["far"].append({"node":self.far_coeff(hsum/count.clamp_min(1)),
                "position":self.far_position(level["xy"].to(h.dtype)),
                "base":_degree_weights(i,j,active,level["side"]**2,h.dtype),
                "alpha":1/max(1,num_interacting) if i.numel() else 0.})
        return state

    @staticmethod
    def _coefficients(node, position, i, j, base, start, stop, width):
        ii=i[...,start:stop];jj=j[...,start:stop]
        left,right=_gather(node,ii),_gather(node,jj)
        p=position[start:stop][None,:,:]
        diagonal=torch.tanh(.5*(left[...,:width]+right[...,:width])+p[...,:width])
        offset=torch.tanh(left[...,width:2*width]-right[...,width:2*width]+p[...,width:2*width])
        weight=torch.sigmoid(.5*(left[...,2*width:]+right[...,2*width:])+p[...,2*width:])*base[:,start:stop]
        return ii,jj,diagonal,offset,weight

    def _factors(self, z, node, position, i, j, base, gradient=True):
        """Tile pair residuals; shared scalar-energy definition and exact adjoints."""
        result=torch.zeros_like(z) if gradient else z.new_zeros(z.shape[0])
        for start in range(0,i.shape[-1],self.edge_tile):
            ii,jj,d,b,w=self._coefficients(node,position,i,j,base,start,start+self.edge_tile,z.shape[-1])
            residual=_gather(z,ii)-d*_gather(z,jj)-b
            if gradient:
                force=w*residual
                if ii.ndim==1:
                    ii=ii[None,:].expand(z.shape[0],-1);jj=jj[None,:].expand(z.shape[0],-1)
                result.scatter_add_(1,ii[...,None].expand_as(force),force)
                result.scatter_add_(1,jj[...,None].expand_as(force),-d*force)
            else:
                result=result+.5*(w*residual.square()).sum((1,2))
        return result

    def energy_gradient(self, z, state):
        h,valid=state["h"],state["valid"]
        if self.variant=="anchor":
            return (z-h)*valid[...,None]
        gradient=z-h
        if self.near_strength:
            near=self._factors(z,state["near_node"],state["near_position"],state["near_i"],state["near_j"],state["near_base"])
            gradient=gradient+self.near_strength*near
        if self.far_strength:
            projected=F.linear(z,state["projection"])*valid[...,None]
            sums=self._tree_sums(projected,state)
            contributions=[]
            for level,count,sums_at_level,factors in zip(state["geometry"]["levels"],state["counts"],sums,state["far"]):
                means=sums_at_level/count.clamp_min(1)
                own=self._factors(means,factors["node"],factors["position"],level["i"],level["j"],factors["base"])
                contributions.append(factors["alpha"]*own/count.clamp_min(1))
            down=contributions[0]
            for level,own in zip(state["geometry"]["levels"][1:],contributions[1:]):
                down=down.index_select(1,level["parent"])+own
            far=_gather(down,state["leaf_index"])@state["projection"]
            gradient=gradient+self.far_strength*far
        return gradient*valid[...,None]

    def energy(self,z,state,return_terms=False):
        valid=state["valid"]
        anchor=.5*((z-state["h"]).square()*valid[...,None]).sum((1,2))
        near=anchor*0;far=anchor*0
        if self.variant!="anchor":
            near=self.near_strength*self._factors(z,state["near_node"],state["near_position"],state["near_i"],state["near_j"],state["near_base"],False)
            sums=self._tree_sums(F.linear(z,state["projection"])*valid[...,None],state)
            for level,count,s,fac in zip(state["geometry"]["levels"],state["counts"],sums,state["far"]):
                far=far+self.far_strength*fac["alpha"]*self._factors(s/count.clamp_min(1),fac["node"],fac["position"],level["i"],level["j"],fac["base"],False)
        terms={"anchor":anchor,"near":near,"far":far,"total":anchor+near+far}
        return terms if return_terms else terms["total"]

    def refine(self,h,state,return_trace=False):
        z=h
        trace=[self.energy(z,state).detach()] if return_trace else None
        if self.variant!="anchor":
            for _ in range(self.refinement_steps):
                if self.checkpoint_refinement and torch.is_grad_enabled() and h.requires_grad:
                    grad=checkpoint(self.energy_gradient,z,state,use_reentrant=False,preserve_rng_state=False)
                else:
                    grad=self.energy_gradient(z,state)
                z=(z-self.step_size*grad)*state["valid"][...,None]
                if return_trace: trace.append(self.energy(z,state).detach())
        return (z,torch.stack(trace)) if return_trace else z

    def embed(self,images,indices=None,valid=None):
        if images.ndim!=4 or images.shape[1]!=3:
            raise ValueError("images must have [B,3,H,W] shape")
        batch,_,height,width=images.shape
        if height%self.patch_size or width%self.patch_size:
            raise ValueError("Image dimensions must be divisible by patch_size")
        grid=(height//self.patch_size,width//self.patch_size);count=grid[0]*grid[1]
        if indices is None:
            if valid is not None: raise ValueError("valid requires indices")
            indices=torch.arange(count,device=images.device).expand(batch,-1)
        if valid is None: valid=torch.ones_like(indices,dtype=torch.bool)
        if indices.device!=images.device or valid.device!=images.device:
            raise ValueError("Images, indices and valid must share a device")
        validate_positions(indices,valid,batch,count)
        raw=F.unfold(images,kernel_size=self.patch_size,stride=self.patch_size).transpose(1,2)
        pixels=_gather(raw,indices.clamp(0,count-1))
        pixels=torch.where(valid[...,None],pixels,torch.zeros_like(pixels))
        h=self.patch_embed(pixels)+position_features(indices.clamp(0,count-1),grid,self.dim,pixels.dtype)
        h=torch.where(valid[...,None],h,torch.zeros_like(h))
        return h,indices,valid,grid

    def forward(self,images,indices=None,valid=None):
        # The theorem concerns real arithmetic; use FP32 (FP64 for audits), even
        # when the surrounding JEPA predictor uses automatic mixed precision.
        with torch.autocast(device_type=images.device.type,enabled=False):
            images=images.to(self.patch_embed.weight.dtype)
            h,indices,valid,grid=self.embed(images,indices,valid)
            if self.variant=="anchor":
                z=h
            else:
                state=self.prepare_energy(h,indices,valid,grid)
                z=self.refine(h,state)
            return self.norm(z)*valid[...,None]


class EnergyJEPA(JEPA):
    """Replace both encoder branches while retaining JEPA loss, predictor and EMA."""
    def __init__(self,config,variant=None):
        nn.Module.__init__(self)
        self.config=copy.deepcopy(config)
        e=config.get("energy07",{})
        variant=variant or e.get("variant","energy")
        def option(name,default):
            return e.get(name,config.get("energy_"+name,default))
        self.context=EnergyEncoder(patch_size=config.get("patch_size",8),dim=config.get("dim",192),
            summary_dim=option("summary_dim",32),leaf_size=config.get("leaf_size",2),
            refinement_steps=option("steps",4),near_strength=option("near_strength",.25),far_strength=option("far_strength",.25),
            step_size=option("step_size",.5),coefficient_dim=option("coefficient_dim",32),
            checkpoint_refinement=option("checkpoint",True),edge_tile=option("edge_tile",256),variant=variant)
        self.target=copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor=Predictor(self.context.dim,dim=config.get("predictor_dim",32),depth=config.get("predictor_depth",1),
            heads=config.get("predictor_heads",config.get("heads",4)))


def build_model(config,kind="semantic",variant=None):
    if kind=="vit":
        cfg=copy.deepcopy(config);cfg["kind"]="vit"
        return JEPA(cfg)
    return EnergyJEPA(config,variant=variant)
